In [ ]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe

from utils.melt_stage_writer import (
    build_sensor_messages_stage,
    validate_sensor_messages_stage,
)


In [ ]:

engine = get_engine_from_env()


: 

In [ ]:

melt_table_name = build_sensor_messages_stage(
    engine=engine,
    schema="capstone",
    source_table="synthetic_observations_premelt_stage",
    target_table="synthetic_sensor_messages_stage",
    if_exists="replace",
    random_seed=42,
    n_sensors=52,
)


In [ ]:

print("Built table:", melt_table_name)


In [ ]:

validation_dataframe = validate_sensor_messages_stage(
    engine=engine,
    schema="capstone",
    table_name="synthetic_sensor_messages_stage",
)


In [ ]:
display(validation_dataframe)

----

In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        generated_row_id,
        sensor_name,
        sensor_index,
        sensor_value,
        message_sequence_index,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_sensor_messages_stage
    ORDER BY observation_index, message_sequence_index
    LIMIT 104
    """
)


In [ ]:

display(sample_dataframe)

In [ ]:
sequence_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        COUNT(*) AS message_count,
        COUNT(DISTINCT sensor_index) AS distinct_sensor_count,
        MIN(message_sequence_index) AS min_msg_seq,
        MAX(message_sequence_index) AS max_msg_seq,
        COUNT(DISTINCT message_sequence_index) AS distinct_msg_seq_count
    FROM capstone.synthetic_sensor_messages_stage
    GROUP BY observation_index
    ORDER BY observation_index
    LIMIT 10
    """
)

display(sequence_check_dataframe)